# Union de las dos descargas: capturas y actuaciones

Las dos fuentes vienen partidas en dos archivos. `capturas_with_status` y
`actuaciones` cubren de 2006 a marzo de 2019; `capturas_with_status2` y
`actuaciones2` cubren de 2019 a 2026.

En capturas el 2019 aparece en los dos archivos y la version valida es la
del segundo, asi que del primero se corta todo lo que sea del 2019 en
adelante y del segundo se deja hasta julio de 2026 inclusive. Las
actuaciones se concatenan completas.

Salidas: `capturas_2006_2026.parquet` y `actuaciones_2006_2026.parquet`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

DESCARGAS = Path.home() / "Downloads"

RUTA_1 = DESCARGAS / "actuaciones.parquet"
RUTA_2 = DESCARGAS / "actuaciones2.parquet"
RUTA_SALIDA = DESCARGAS / "actuaciones_2006_2026.parquet"
RUTA_3 = DESCARGAS / "capturas_with_status.parquet"
RUTA_4 = DESCARGAS / "capturas_with_status2.parquet"
RUTA_SALIDA2 = DESCARGAS / "capturas_2006_2026.parquet"

CORTE = pd.Timestamp("2019-01-01")

TOPE = pd.Timestamp("2026-08-01")

COL_FECHA = None

CONSERVAR_SIN_FECHA = False

## 1. Esquema de los dos archivos de capturas

In [ ]:
esquema_1 = pq.read_schema(RUTA_3)
esquema_2 = pq.read_schema(RUTA_4)

meta_1 = pq.ParquetFile(RUTA_3).metadata
meta_2 = pq.ParquetFile(RUTA_4).metadata

print(f"capturas  : {meta_1.num_rows:,} filas x "
      f"{len(esquema_1.names)} columnas")
print(f"capturas2 : {meta_2.num_rows:,} filas x "
      f"{len(esquema_2.names)} columnas")


cols_1, cols_2 = set(esquema_1.names), set(esquema_2.names)

solo_1 = sorted(cols_1 - cols_2)
solo_2 = sorted(cols_2 - cols_1)

if solo_1:
    print(f"\nSolo en capturas.parquet  ({len(solo_1)}): {solo_1}")
if solo_2:
    print(f"\nSolo en capturas2.parquet ({len(solo_2)}): {solo_2}")
if not solo_1 and not solo_2:
    print("\nMismas columnas en los dos archivos.")


distintos = [
    (c, str(esquema_1.field(c).type), str(esquema_2.field(c).type))
    for c in sorted(cols_1 & cols_2)
    if esquema_1.field(c).type != esquema_2.field(c).type
]
if distintos:
    print("\nColumnas con tipo distinto entre archivos:\n")
    print(pd.DataFrame(distintos,
                       columns=["columna", "archivo 1", "archivo 2"]
                       ).to_string(index=False))
    print("\nAl concatenar, esas columnas quedan en object. Si alguna "
          "importa, conviene convertirla a mano antes de guardar.")


print("\nColumnas que parecen de fecha:\n")
for c in esquema_1.names:
    if "fecha" in c.lower() or "date" in c.lower():
        print(f"  {c:<35} {esquema_1.field(c).type}")

## 2. Columna de fecha

In [ ]:
CANDIDATOS_FECHA = ["FECHA"]


def resolver_fecha(nombres):
    if COL_FECHA is not None:
        if COL_FECHA not in nombres:
            raise KeyError(f"'{COL_FECHA}' no está en el archivo")
        return COL_FECHA

    normal = {n.lower().replace(" ", "").replace("_", ""): n for n in nombres}

    for candidato in CANDIDATOS_FECHA:
        clave = candidato.lower().replace(" ", "").replace("_", "")
        if clave in normal:
            return normal[clave]

    de_fecha = [n for n in nombres if "fecha" in n.lower()]
    if len(de_fecha) == 1:
        return de_fecha[0]

    raise KeyError(
        f"No pude elegir sola la columna de fecha. Candidatas: {de_fecha}. "
        "Escribí la correcta en COL_FECHA."
    )


COL_FECHA = resolver_fecha(esquema_1.names)

if COL_FECHA not in esquema_2.names:
    raise KeyError(f"'{COL_FECHA}' existe en el archivo 1 pero no en el 2")


print(f"Columna de fecha: {COL_FECHA}")

## 3. Carga y recorte del traslape

Se imprime cuanto se descarta en cada archivo para verificar que el
traslape era el esperado.

In [ ]:
def cargar(ruta):
    df = pd.read_parquet(ruta)
    df[COL_FECHA] = pd.to_datetime(df[COL_FECHA], errors="coerce")
    return df


def resumen_fechas(df, etiqueta):
    fecha = df[COL_FECHA]
    sin_fecha = int(fecha.isna().sum())
    print(f"{etiqueta}: {len(df):,} filas   "
          f"rango {fecha.min():%Y-%m-%d} a {fecha.max():%Y-%m-%d}   "
          f"sin fecha: {sin_fecha:,}")
    return sin_fecha


df1 = cargar(RUTA_3)
df2 = cargar(RUTA_4)

print("Como vienen:\n")
sin_fecha_1 = resumen_fechas(df1, "capturas ")
sin_fecha_2 = resumen_fechas(df2, "capturas2")

mantener_1 = df1[COL_FECHA].lt(CORTE)
if CONSERVAR_SIN_FECHA:
    mantener_1 |= df1[COL_FECHA].isna()


descarte_1 = int((~mantener_1).sum())
print(f"\nArchivo 1: se descartan {descarte_1:,} filas "
      f"({descarte_1 / len(df1):.1%}) por ser del {CORTE:%Y-%m-%d} en adelante")


if descarte_1:
    por_anio = (df1.loc[~mantener_1, COL_FECHA].dt.year
                .value_counts(dropna=False).sort_index())
    print("  descartadas por año:")
    print("   ", por_anio.to_dict())


mantener_2 = df2[COL_FECHA].ge(CORTE) & df2[COL_FECHA].lt(TOPE)
if CONSERVAR_SIN_FECHA:
    mantener_2 |= df2[COL_FECHA].isna()


antes_corte = int(df2[COL_FECHA].lt(CORTE).sum())
pasado_tope = int(df2[COL_FECHA].ge(TOPE).sum())

print(f"\nArchivo 2: se descartan {antes_corte:,} filas anteriores al corte "
      f"y {pasado_tope:,} posteriores a julio de 2026")


df1 = df1.loc[mantener_1].copy()
df2 = df2.loc[mantener_2].copy()

print("\nDespués del recorte:\n")
resumen_fechas(df1, "capturas ")
resumen_fechas(df2, "capturas2")

## 4. Union de capturas y guardado

`origen` queda como columna nueva para rastrear de que archivo salio cada
fila. El chequeo de duplicados es informativo y no borra nada.

In [ ]:
df1["origen"] = "capturas"
df2["origen"] = "capturas2"

union = pd.concat([df1, df2], ignore_index=True, sort=False)
union = union.sort_values(COL_FECHA, kind="mergesort").reset_index(drop=True)

print(f"Unión: {len(union):,} filas x {len(union.columns)} columnas")
print(f"Suma de las partes: {len(df1):,} + {len(df2):,} = "
      f"{len(df1) + len(df2):,}")


print("\nFilas por origen:")
print(union["origen"].value_counts().to_string())

print("\nFilas por año:\n")
por_anio = union[COL_FECHA].dt.year.value_counts(dropna=False).sort_index()
print(por_anio.to_string())

traslape = union.loc[union[COL_FECHA].dt.year.eq(2019), "origen"].value_counts()
print("\n2019 por origen (debe aparecer solo capturas2):")
print(traslape.to_string())

repetidas = int(union.duplicated().sum())
print(f"\nFilas idénticas repetidas: {repetidas:,}")
if repetidas:
    print("  No se borran. Revisá si son legítimas antes de decidir.")


columnas_vacias = union.columns[union.isna().all()].tolist()
if columnas_vacias:
    print(f"\nColumnas que quedaron 100% vacías: {columnas_vacias}")


union.to_parquet(RUTA_SALIDA2, index=False)
print(f"\nGuardado: {RUTA_SALIDA2}")

## 5. Union de actuaciones

Concatenacion directa de los dos archivos, sin recorte.

In [ ]:
df3 = pd.read_parquet(RUTA_1)
df4 = pd.read_parquet(RUTA_2)

print(f"actuaciones  : {len(df3):,} filas x {len(df3.columns)} columnas")
print(f"actuaciones2 : {len(df4):,} filas x {len(df4.columns)} columnas")

solo_1 = sorted(set(df3.columns) - set(df4.columns))
solo_2 = sorted(set(df4.columns) - set(df3.columns))
if solo_1:
    print(f"\nSolo en actuaciones  : {solo_1}")
if solo_2:
    print(f"Solo en actuaciones2 : {solo_2}")


df3["origen"] = "actuaciones"
df4["origen"] = "actuaciones2"

union = pd.concat([df3, df4], ignore_index=True, sort=False)

print(f"\nUnión: {len(union):,} filas x {len(union.columns)} columnas")
print(f"Suma de las partes: {len(df3):,} + {len(df4):,} = {len(df3) + len(df4):,}")
print("\nFilas por origen:")
print(union["origen"].value_counts().to_string())

union.to_parquet(RUTA_SALIDA, index=False)
print(f"\nGuardado: {RUTA_SALIDA}")